# Évaluation du pipeline RAG avec RAGAS

**Objectif :** mesurer objectivement la qualité du pipeline via 4 métriques automatisées.

| Métrique | Ce qu'elle mesure | Données requises |
|---|---|---|
| `faithfulness` | La réponse reste-t-elle fidèle aux documents récupérés ? | question, answer, contexts |
| `answer_relevancy` | La réponse répond-elle bien à la question posée ? | question, answer, contexts |
| `context_precision` | Les chunks récupérés sont-ils tous utiles (pas de bruit) ? | question, contexts, ground_truth |
| `context_recall` | Les bons documents ont-ils été retrouvés ? | contexts, ground_truth |

Les 4 métriques utilisent Mistral comme **LLM-as-judge** — Mistral évalue lui-même la qualité des réponses.

Étapes :
1. Chargement de l'index et des clients
2. Définition du jeu de test (questions + réponses de référence)
3. Collecte des réponses RAG et contextes récupérés
4. Évaluation RAGAS
5. Analyse des résultats

In [26]:
import os
import pandas as pd
from datasets import Dataset
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings
from langchain_mistralai.chat_models import ChatMistralAI
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

embed_model = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)
faiss_store = FAISS.load_local("../data/faiss_index", embed_model, allow_dangerous_deserialization=True)
client = MistralClient(api_key=os.getenv("MISTRAL_API_KEY"))

print(f"Index chargé : {faiss_store.index.ntotal} vecteurs")

Index chargé : 5568 vecteurs


## 1. Jeu de test annoté

RAGAS a besoin de deux choses annotées manuellement :
- `question` — une question représentative d'un vrai usage
- `ground_truth` — la réponse de référence qu'on attendrait idéalement

Les deux autres colonnes (`answer`, `contexts`) seront générées automatiquement par notre pipeline.

**Sur les `ground_truth`** : pour `context_precision` et `context_recall`, RAGAS vérifie si les contextes récupérés contiennent assez d'information pour produire la `ground_truth`. Ces réponses sont ici volontairement génériques — pour une évaluation rigoureuse en production, il faudrait les annoter à partir des vraies données.

In [ ]:
test_questions = [
    "Quels concerts sont prévus à Rennes ?",
    "Y a-t-il des expositions gratuites à Rennes ?",
    "Quels événements sont proposés pour les enfants à Rennes ?",
    "Y a-t-il des spectacles de danse à Rennes ?",
    "Quels événements gratuits peut-on trouver à Rennes ?",
    "Y a-t-il des conférences ou ateliers culturels à Rennes ?",
    "Y a-t-il des événements musicaux en plein air à Rennes ?",
    "Quels spectacles de théâtre sont programmés à Rennes ?",
    "Y a-t-il des projections de cinéma à Rennes ?",
    "Quels festivals ont lieu à Rennes ?",
    "Y a-t-il des visites guidées à Rennes ?",
    "Quels événements de musique classique sont prévus à Rennes ?",
    "Y a-t-il des événements familiaux à Rennes ?",
    "Quels musées proposent des expositions à Rennes ?",
    "Y a-t-il des événements gratuits pour les enfants à Rennes ?",
    "Y a-t-il des ateliers créatifs ou artistiques à Rennes ?",
    "Quels événements de danse contemporaine sont programmés à Rennes ?",
    "Y a-t-il des lectures ou rencontres littéraires à Rennes ?",
    "Quels événements se déroulent au Théâtre National de Bretagne ?",
    "Y a-t-il des concerts gratuits à Rennes ?",
    "Quels événements de cirque ou arts de la rue sont programmés à Rennes ?",
    "Y a-t-il des expositions d'art contemporain à Rennes ?",
    "Quels événements se déroulent aux Champs Libres à Rennes ?",
    "Y a-t-il des événements de musique électronique à Rennes ?",
    "Y a-t-il des opéras ou spectacles lyriques à Rennes ?",
    "Quels événements nocturnes ou soirées culturelles sont prévus à Rennes ?",
    "Y a-t-il des concerts de jazz à Rennes ?",
    "Quels événements sont proposés pour les adolescents à Rennes ?",
    "Y a-t-il des marchés ou événements de rue à Rennes ?",
    "Quels événements liés à la gastronomie ou à la culture locale sont prévus à Rennes ?",
]

ground_truths = [
    "Plusieurs concerts sont prévus à Rennes avec des informations sur les dates, les lieux et les tarifs.",
    "Oui, il existe des expositions gratuites à Rennes dans des musées et centres d'art de la ville.",
    "Des activités pour enfants sont proposées à Rennes, notamment des ateliers, jeux et visites guidées.",
    "Des spectacles de danse sont organisés à Rennes dans différentes salles culturelles.",
    "De nombreux événements gratuits sont disponibles à Rennes dans différents lieux culturels.",
    "Des conférences et ateliers culturels sont organisés à Rennes.",
    "Des événements musicaux en plein air sont organisés à Rennes.",
    "Des spectacles de théâtre sont programmés à Rennes dans différentes salles.",
    "Des projections cinématographiques sont organisées à Rennes.",
    "Des festivals culturels ont lieu à Rennes sur différentes thématiques.",
    "Des visites guidées sont proposées à Rennes pour découvrir la ville et ses monuments.",
    "Des concerts de musique classique sont prévus à Rennes.",
    "Des événements familiaux sont proposés à Rennes, adaptés aux enfants et aux parents.",
    "Plusieurs musées de Rennes proposent des expositions temporaires et permanentes.",
    "Des événements gratuits pour les enfants sont proposés à Rennes.",
    "Des ateliers créatifs et artistiques sont proposés à Rennes pour différents publics.",
    "Des événements de danse contemporaine sont programmés à Rennes.",
    "Des lectures et rencontres avec des auteurs sont organisées à Rennes.",
    "Le Théâtre National de Bretagne accueille des spectacles et événements culturels à Rennes.",
    "Des concerts gratuits sont proposés à Rennes dans différents lieux.",
    "Des spectacles de cirque et d'arts de la rue sont programmés à Rennes.",
    "Des expositions d'art contemporain sont présentées dans les galeries et musées de Rennes.",
    "Les Champs Libres accueillent des événements culturels variés à Rennes.",
    "Des événements de musique électronique sont programmés à Rennes.",
    "Des opéras ou spectacles lyriques sont présentés à Rennes.",
    "Des soirées et événements nocturnes culturels sont proposés à Rennes.",
    "Des concerts de jazz sont programmés à Rennes.",
    "Des événements destinés aux adolescents sont proposés à Rennes.",
    "Des marchés et événements de rue sont organisés à Rennes.",
    "Des événements liés à la gastronomie et à la culture locale sont organisés à Rennes.",
]

print(f"{len(test_questions)} questions de test définies")

## 2. Collecte des réponses et contextes

RAGAS a besoin de deux colonnes générées par le pipeline :
- `answer` — la réponse produite par Mistral
- `contexts` — la **liste des textes bruts** récupérés par FAISS (les `page_content`)

La fonction `ask_with_context()` est une version de `ask()` qui retourne les deux. Les `contexts` sont les corpus tels qu'indexés — c'est sur eux que RAGAS évalue faithfulness et context_recall.

In [ ]:
def ask(question: str, k: int = 5) -> tuple[str, list[str]]:
    """Pipeline RAG complet. Retourne (réponse générée, textes complets fournis au modèle)."""
    docs = faiss_store.similarity_search(question, k=k)

    context_parts = []
    for doc in docs:
        m = doc.metadata
        context_parts.append(
            f"Titre : {m['title']}\n"
            f"Description : {doc.page_content}\n"
            f"Tarif : {m['conditions']}\n"
            f"Dates : {m['date_start']} → {m['date_end']}\n"
            f"Lieu : {m['location']}\n"
            f"Lien : {m['url']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""Tu es un assistant spécialisé dans les événements culturels à Rennes.
Réponds à la question en te basant uniquement sur les événements fournis ci-dessous.
Si l'information n'est pas dans le contexte, dis-le clairement.

ÉVÉNEMENTS :
{context}

QUESTION : {question}

RÉPONSE :"""

    response = client.chat(
        model="mistral-small-latest",
        messages=[ChatMessage(role="user", content=prompt)]
    )
    # context_parts = texte complet fourni au modèle (description + métadonnées)
    # RAGAS doit évaluer la fidélité par rapport à ce qui a réellement été donné au LLM
    return response.choices[0].message.content, context_parts


answers = []
contexts_list = []

for i, question in enumerate(test_questions):
    print(f"[{i+1}/{len(test_questions)}] {question}")
    answer, contexts = ask(question)
    answers.append(answer)
    contexts_list.append(contexts)

print("\nCollecte terminée.")

## 3. Construction du dataset RAGAS

RAGAS attend un objet `Dataset` de HuggingFace avec exactement 4 colonnes : `question`, `answer`, `contexts`, `ground_truth`. On assemble ici les données collectées.

In [29]:
data = pd.DataFrame({
    "question": test_questions,
    "answer": answers,
    "contexts": contexts_list,
    "ground_truth": ground_truths,
})

In [30]:
data.head()

,question,answer,contexts,ground_truth
0,Quels concerts sont prévus à Rennes ?,Voici les concerts prévus à Rennes selon les é...,[Titre : Concert : musique italienne du XVIIèm...,Plusieurs concerts sont prévus à Rennes avec d...
1,Y a-t-il des expositions gratuites à Rennes ?,"Oui, il y a des expositions gratuites à Rennes...",[Titre : Musées gratuits!\nDescription : Musée...,"Oui, il existe des expositions gratuites à Ren..."
2,Quels événements sont proposés pour les enfant...,Voici les événements proposés pour les enfants...,[Titre : Vues sur Rennes\nDescription : Vues s...,Des activités pour enfants sont proposées à Re...
3,Y a-t-il des spectacles de danse à Rennes ?,"Oui, il y a plusieurs spectacles de danse à Re...",[Titre : DANSES TOUS STYLES : LES RENCONTRES C...,Des spectacles de danse sont organisés à Renne...
4,Quels événements gratuits peut-on trouver à Re...,Voici les événements gratuits à Rennes mention...,[Titre : Vues sur Rennes\nDescription : Vues s...,De nombreux événements gratuits sont disponibl...


In [31]:
ragas_dataset = Dataset.from_pandas(data)

print(ragas_dataset)

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 30
})


## 4. Évaluation RAGAS

RAGAS utilise un LLM pour évaluer automatiquement les réponses (**LLM-as-judge**). Par défaut il appelle OpenAI — on le reconfigure ici pour utiliser Mistral, cohérent avec notre stack.

> Cette cellule fait plusieurs appels API par question × par métrique.

In [32]:
ragas_llm = ChatMistralAI(
    model="mistral-small-latest",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)

ragas_embeddings = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)

result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

print(result)

Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

{'faithfulness': 0.9054, 'answer_relevancy': 0.8874, 'context_precision': 0.7750, 'context_recall': 1.0000}


## 5. Résultats et analyse

In [33]:
# Détail par question
df = result.to_pandas()
df["question"] = df["question"].str[:55]
display(df[["question", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]].round(3))

,question,faithfulness,answer_relevancy,context_precision,context_recall
0,Quels concerts sont prévus à Rennes ?,0.737,0.960,0.500,1.0
1,Y a-t-il des expositions gratuites à Rennes ?,0.800,0.866,1.000,1.0
2,Quels événements sont proposés pour les enfant...,1.000,0.878,1.000,1.0
3,Y a-t-il des spectacles de danse à Rennes ?,1.000,0.826,1.000,1.0
4,Quels événements gratuits peut-on trouver à Re...,1.000,0.940,1.000,1.0
5,Y a-t-il des conférences ou ateliers culturels...,0.895,0.838,1.000,1.0
6,Y a-t-il des événements musicaux en plein air ...,1.000,0.853,1.000,1.0
7,Quels spectacles de théâtre sont programmés à ...,1.000,0.926,0.000,1.0
8,Y a-t-il des projections de cinéma à Rennes ?,1.000,0.862,1.000,1.0
9,Quels festivals ont lieu à Rennes ?,0.000,0.940,0.000,1.0
